# Calculate Errors

Calculate MAE and RMSE for the two prediction rules.

Open this notebook in `notebooks/M4/`. Run the cells in order. Saving again replaces this notebook's output files.


## Shared folder setup

In [1]:
from pathlib import Path
PROJECT_ROOT = Path("../..").resolve()
PROJECT_ROOT

WindowsPath('C:/Users/mo123/Desktop/household-energy-prediction/household-energy-prediction')

## The five-row metric example

In [2]:
from pathlib import Path
import pandas as pd
example = pd.DataFrame({
    "actual_kwh": [1.0, 0.5, 0.8, float("nan"), 0.9],
    "pred_prev_hour": [0.6, 0.5, 1.2, 0.4, float("nan")],
    "pred_prev_day": [0.8, 0.2, 0.8, 0.3, 0.7]})
example

,actual_kwh,pred_prev_hour,pred_prev_day
0,1.0,0.6,0.8
1,0.5,0.5,0.2
2,0.8,1.2,0.8
3,NaN,0.4,0.3
4,0.9,NaN,0.7


## Find missing actual values

In [3]:
example_missing_target = example["actual_kwh"].isna()
example_missing_target

0    False
1    False
2    False
3     True
4    False
Name: actual_kwh, dtype: bool

## Find missing predictions without double-counting

In [4]:
example_missing_prediction = ~example_missing_target & example[["pred_prev_hour",
    "pred_prev_day"]].isna().any(axis=1)
example_missing_prediction

0    False
1    False
2    False
3    False
4     True
dtype: bool

## Keep a common comparison set

In [5]:
example_valid = ~example_missing_target & ~example_missing_prediction
example_used = example[example_valid]
example_used

,actual_kwh,pred_prev_hour,pred_prev_day
0,1.0,0.6,0.8
1,0.5,0.5,0.2
2,0.8,1.2,0.8


## Calculate signed errors

In [6]:
example_errors = pd.DataFrame({
    "previous_hour": example_used["actual_kwh"] - example_used["pred_prev_hour"],
    "previous_day": example_used["actual_kwh"] - example_used["pred_prev_day"]})
example_errors

,previous_hour,previous_day
0,0.4,0.2
1,0.0,0.3
2,-0.4,0.0


## Remove the signs

In [7]:
example_absolute = example_errors.abs()
example_absolute

,previous_hour,previous_day
0,0.4,0.2
1,0.0,0.3
2,0.4,0.0


## Calculate MAE

In [8]:
example_mae = example_absolute.mean()
example_mae

previous_hour    0.266667
previous_day     0.166667
dtype: float64

## Square the errors

In [9]:
example_squared = example_errors ** 2
example_squared

,previous_hour,previous_day
0,0.16,0.04
1,0.00,0.09
2,0.16,0.00


## Calculate RMSE

In [10]:
example_rmse = example_squared.mean() ** 0.5
example_rmse

previous_hour    0.326599
previous_day     0.208167
dtype: float64

## Collect the hand-example results

In [11]:
example_scores = pd.DataFrame({"mae_kwh": example_mae, "rmse_kwh": example_rmse})
example_scores.index.name = "rule"
example_scores["n_total"] = len(example)
example_scores["n_missing_target"] = example_missing_target.sum()
example_scores["n_missing_prediction"] = example_missing_prediction.sum()
example_scores["n_scored"] = example_valid.sum()
example_scores

,mae_kwh,rmse_kwh,n_total,n_missing_target,n_missing_prediction,n_scored
rule,,,,,,
previous_hour,0.266667,0.326599,5,1,1,3
previous_day,0.166667,0.208167,5,1,1,3


## Choose the predictions

In [12]:
prediction_path = PROJECT_ROOT / "results/baseline_predictions_sample.csv"
predictions = pd.read_csv(prediction_path)
predictions["target_hour"] = pd.to_datetime(predictions["target_hour"], utc=True)
predictions.head()

,task_id,LCLid,target_hour,actual_kwh,pred_prev_hour,pred_prev_day,label
0,M4-W07-T01,MAC000038,2013-01-01 00:00:00+00:00,1.188,NaN,NaN,PROTOTYPE
1,M4-W07-T01,MAC000038,2013-01-01 01:00:00+00:00,0.503,1.188,NaN,PROTOTYPE
2,M4-W07-T01,MAC000038,2013-01-01 02:00:00+00:00,0.164,0.503,NaN,PROTOTYPE
3,M4-W07-T01,MAC000038,2013-01-01 03:00:00+00:00,0.143,0.164,NaN,PROTOTYPE
4,M4-W07-T01,MAC000038,2013-01-01 04:00:00+00:00,0.128,0.143,NaN,PROTOTYPE


## See the supplied scope

In [13]:
predictions["LCLid"].unique(), len(predictions), predictions["target_hour"].min(), predictions["target_hour"].max()

(array(['MAC000038', 'MAC000045', 'MAC001617'], dtype=object),
 504,
 Timestamp('2013-01-01 00:00:00+0000', tz='UTC'),
 Timestamp('2013-01-07 23:00:00+0000', tz='UTC'))

## Count missing actual values

In [14]:
missing_target = predictions["actual_kwh"].isna()
missing_target.sum()

6

## Count rows missing either prediction

In [15]:
missing_prediction = ~missing_target & predictions[["pred_prev_hour",
    "pred_prev_day"]].isna().any(axis=1)
missing_prediction.sum()

80

## Use the same rows for both baselines

In [16]:
valid = ~missing_target & ~missing_prediction
used = predictions[valid]
len(used), used.head()

(418,
        task_id      LCLid               target_hour  actual_kwh  \
 24  M4-W07-T01  MAC000038 2013-01-02 00:00:00+00:00       0.227   
 25  M4-W07-T01  MAC000038 2013-01-02 01:00:00+00:00       0.111   
 26  M4-W07-T01  MAC000038 2013-01-02 02:00:00+00:00       0.136   
 27  M4-W07-T01  MAC000038 2013-01-02 03:00:00+00:00       0.131   
 28  M4-W07-T01  MAC000038 2013-01-02 04:00:00+00:00       0.144   
 
     pred_prev_hour  pred_prev_day      label  
 24           0.645          1.188  PROTOTYPE  
 25           0.227          0.503  PROTOTYPE  
 26           0.111          0.164  PROTOTYPE  
 27           0.136          0.143  PROTOTYPE  
 28           0.131          0.128  PROTOTYPE  )

## Show the scoring counts

In [17]:
row_counts = pd.Series({"n_total": len(predictions), "n_missing_target": missing_target.sum(),
                        "n_missing_prediction": missing_prediction.sum(), "n_scored": valid.sum()})
row_counts

n_total                 504
n_missing_target          6
n_missing_prediction     80
n_scored                418
dtype: int64

## Calculate both errors on the common rows

In [18]:
errors = pd.DataFrame({
    "previous_hour": used["actual_kwh"] - used["pred_prev_hour"],
    "previous_day": used["actual_kwh"] - used["pred_prev_day"]})
errors.head()

,previous_hour,previous_day
24,-0.418,-0.961
25,-0.116,-0.392
26,0.025,-0.028
27,-0.005,-0.012
28,0.013,0.016


## Calculate the prototype MAE

In [19]:
mae_values = errors.abs().mean()
mae_values

previous_hour    0.304057
previous_day     0.310679
dtype: float64

## Calculate the prototype RMSE

In [20]:
rmse_values = (errors ** 2).mean() ** 0.5
rmse_values

previous_hour    0.507470
previous_day     0.520032
dtype: float64

## Build the score table

In [21]:
scores = pd.DataFrame({"mae_kwh": mae_values, "rmse_kwh": rmse_values})
scores.index.name = "rule"
scores = scores.reset_index()
for column, value in row_counts.items():
    scores[column] = value
scores

,rule,mae_kwh,rmse_kwh,n_total,n_missing_target,n_missing_prediction,n_scored
0,previous_hour,0.304057,0.507470,504,6,80,418
1,previous_day,0.310679,0.520032,504,6,80,418


## Keep the task and split labels

In [22]:
scores.insert(0, "task_id", "M4-W07-T02")
scores.insert(1, "split", "prototype")
scores

,task_id,split,rule,mae_kwh,rmse_kwh,n_total,n_missing_target,n_missing_prediction,n_scored
0,M4-W07-T02,prototype,previous_hour,0.304057,0.507470,504,6,80,418
1,M4-W07-T02,prototype,previous_day,0.310679,0.520032,504,6,80,418


## Read the results schema

In [23]:
schema = pd.read_csv(PROJECT_ROOT / "notebooks/M4/score_columns.csv")
schema

,column,type,meaning
0,task_id,text,task that produced the scores
1,split,text,prototype / validation / test
2,rule,text,baseline or model name
3,n_total,int,all rows
4,n_missing_target,int,rows removed because the actual is missing
5,n_missing_prediction,int,rows removed because a prediction is missing
6,n_scored,int,rows used for scoring (same for all rules)
7,mae_kwh,float,MAE in kWh
8,rmse_kwh,float,RMSE in kWh


## Order the columns

In [24]:
scores = scores[schema["column"]].copy()
scores

,task_id,split,rule,n_total,n_missing_target,n_missing_prediction,n_scored,mae_kwh,rmse_kwh
0,M4-W07-T02,prototype,previous_hour,504,6,80,418,0.304057,0.507470
1,M4-W07-T02,prototype,previous_day,504,6,80,418,0.310679,0.520032


## Compare the row counts

In [25]:
row_counts["n_total"] == row_counts["n_missing_target"] + row_counts["n_missing_prediction"] + row_counts["n_scored"]

True

## Choose the output folder

In [26]:
output_dir = PROJECT_ROOT / 'results'
output_dir.mkdir(parents=True, exist_ok=True)
output_dir

WindowsPath('C:/Users/mo123/Desktop/household-energy-prediction/household-energy-prediction/results')

## Save the prototype scores

In [27]:
scores_path = output_dir / "baseline_scores_sample.csv"
scores.to_csv(scores_path, index=False)
scores_path

WindowsPath('C:/Users/mo123/Desktop/household-energy-prediction/household-energy-prediction/results/baseline_scores_sample.csv')

## Save the hand-example scores

In [28]:
example_path = output_dir / "error_calculation_example.csv"
example_scores.reset_index().to_csv(example_path, index=False)
example_path

WindowsPath('C:/Users/mo123/Desktop/household-energy-prediction/household-energy-prediction/results/error_calculation_example.csv')

## Save the column descriptions

In [29]:
schema_path = output_dir / "score_columns.csv"
schema.to_csv(schema_path, index=False)
schema_path

WindowsPath('C:/Users/mo123/Desktop/household-energy-prediction/household-energy-prediction/results/score_columns.csv')

## Read back the saved score table

In [30]:
pd.read_csv(scores_path)

,task_id,split,rule,n_total,n_missing_target,n_missing_prediction,n_scored,mae_kwh,rmse_kwh
0,M4-W07-T02,prototype,previous_hour,504,6,80,418,0.304057,0.507470
1,M4-W07-T02,prototype,previous_day,504,6,80,418,0.310679,0.520032


# Week 2: Validation errors

Score both baselines on the saved VALIDATION predictions. These are not final TEST results.

## Read the validation predictions

In [31]:
val_prediction_path = PROJECT_ROOT / "results/validation_predictions.csv"
val = pd.read_csv(val_prediction_path)
val["target_hour"] = pd.to_datetime(val["target_hour"], utc=True)
val["period"].unique(), len(val), val["LCLid"].nunique()

(array(['VALIDATION'], dtype=object), 349127, 240)

## Count missing actual values and predictions

In [32]:
val_missing_target = val["actual_kwh"].isna()
val_missing_prediction = ~val_missing_target & val[["pred_prev_hour",
    "pred_prev_day"]].isna().any(axis=1)
val_missing_target.sum(), val_missing_prediction.sum()

(0, 0)

## Use the same rows for both baselines

In [33]:
val_valid = ~val_missing_target & ~val_missing_prediction
val_used = val[val_valid]
len(val_used)

349127

## Check both methods score the same homes and hours

In [34]:
hour_keys = val_used.loc[val_used["pred_prev_hour"].notna(), ["LCLid", "target_hour"]]
day_keys = val_used.loc[val_used["pred_prev_day"].notna(), ["LCLid", "target_hour"]]
hour_keys.reset_index(drop=True).equals(day_keys.reset_index(drop=True))

True

## Show the scoring counts

In [35]:
val_counts = pd.Series({"n_total": len(val), "n_missing_target": val_missing_target.sum(),
                        "n_missing_prediction": val_missing_prediction.sum(), "n_scored": val_valid.sum()})
val_counts

n_total                 349127
n_missing_target             0
n_missing_prediction         0
n_scored                349127
dtype: int64

## Calculate both errors on the common rows

In [36]:
val_errors = pd.DataFrame({
    "previous_hour": val_used["actual_kwh"] - val_used["pred_prev_hour"],
    "previous_day": val_used["actual_kwh"] - val_used["pred_prev_day"]})
val_errors.head()

,previous_hour,previous_day
0,-0.043,-0.121
1,-0.035,-0.119
2,0.156,0.056
3,-0.110,-0.191
4,-0.019,-0.097


## Calculate the validation MAE and RMSE

In [37]:
val_mae = val_errors.abs().mean()
val_rmse = (val_errors ** 2).mean() ** 0.5
val_mae, val_rmse

(previous_hour    0.174481
 previous_day     0.209382
 dtype: float64,
 previous_hour    0.359841
 previous_day     0.431532
 dtype: float64)

## Build the validation score table

In [38]:
val_scores = pd.DataFrame({"mae_kwh": val_mae, "rmse_kwh": val_rmse})
val_scores.index.name = "rule"
val_scores = val_scores.reset_index()
for column, value in val_counts.items():
    val_scores[column] = value
val_scores.insert(0, "task_id", "M4-W2")
val_scores.insert(1, "split", "validation")
val_scores = val_scores[schema["column"]].copy()
val_scores

,task_id,split,rule,n_total,n_missing_target,n_missing_prediction,n_scored,mae_kwh,rmse_kwh
0,M4-W2,validation,previous_hour,349127,0,0,349127,0.174481,0.359841
1,M4-W2,validation,previous_day,349127,0,0,349127,0.209382,0.431532


## Compare the row counts

In [39]:
val_counts["n_total"] == val_counts["n_missing_target"] + val_counts["n_missing_prediction"] + val_counts["n_scored"]

True

## Save the validation scores

In [40]:
val_scores_path = output_dir / "validation_scores.csv"
val_scores.to_csv(val_scores_path, index=False)
val_scores_path

WindowsPath('C:/Users/mo123/Desktop/household-energy-prediction/household-energy-prediction/results/validation_scores.csv')

## Read back the saved validation scores

In [41]:
pd.read_csv(val_scores_path)

,task_id,split,rule,n_total,n_missing_target,n_missing_prediction,n_scored,mae_kwh,rmse_kwh
0,M4-W2,validation,previous_hour,349127,0,0,349127,0.174481,0.359841
1,M4-W2,validation,previous_day,349127,0,0,349127,0.209382,0.431532


# Week 2: Five-row error check with M6

M6 calculates MAE and RMSE by hand for five VALIDATION rows. M4's formulas are run on the same rows and the results are compared.

## The five rows agreed with M6

In [42]:
five_keys = pd.DataFrame({
    "LCLid": ["MAC000112", "MAC002339", "MAC003520", "MAC003814", "MAC005460"],
    "target_hour": pd.to_datetime(["2013-09-06 16:00", "2013-10-07 17:00", "2013-10-26 15:00",
                                   "2013-09-02 19:00", "2013-09-05 04:00"], utc=True)})
five = five_keys.merge(val, on=["LCLid", "target_hour"], how="left")
five[["LCLid", "target_hour", "period", "actual_kwh", "pred_prev_hour", "pred_prev_day"]]

,LCLid,target_hour,period,actual_kwh,pred_prev_hour,pred_prev_day
0,MAC000112,2013-09-06 16:00:00+00:00,VALIDATION,0.436,0.225,0.182
1,MAC002339,2013-10-07 17:00:00+00:00,VALIDATION,1.720,1.431,0.821
2,MAC003520,2013-10-26 15:00:00+00:00,VALIDATION,0.483,0.326,0.348
3,MAC003814,2013-09-02 19:00:00+00:00,VALIDATION,0.245,0.184,0.290
4,MAC005460,2013-09-05 04:00:00+00:00,VALIDATION,0.197,0.270,0.267


## Check all five rows were found and are scored

In [43]:
len(five) == 5 and five[["actual_kwh", "pred_prev_hour", "pred_prev_day"]].notna().all().all()

True

## M6's hand-style calculation, one row at a time

In [44]:
hand = {}
for rule, column in [("previous_hour", "pred_prev_hour"), ("previous_day", "pred_prev_day")]:
    err = [a - p for a, p in zip(five["actual_kwh"], five[column])]
    hand[rule] = {"mae_kwh": sum(abs(e) for e in err) / len(err),
                  "rmse_kwh": (sum(e * e for e in err) / len(err)) ** 0.5}
hand = pd.DataFrame(hand).T
hand

,mae_kwh,rmse_kwh
previous_hour,0.1582,0.179856
previous_day,0.2806,0.423761


## M4's formulas on the same rows

In [45]:
five_errors = pd.DataFrame({
    "previous_hour": five["actual_kwh"] - five["pred_prev_hour"],
    "previous_day": five["actual_kwh"] - five["pred_prev_day"]})
m4_five = pd.DataFrame({"mae_kwh": five_errors.abs().mean(),
                        "rmse_kwh": (five_errors ** 2).mean() ** 0.5})
m4_five

,mae_kwh,rmse_kwh
previous_hour,0.1582,0.179856
previous_day,0.2806,0.423761


## M6's written results (M6_W2_Task3_five_row_error_check.docx)

In [46]:
m6_written = pd.DataFrame({"mae_kwh": [0.158, 0.281], "rmse_kwh": [0.180, 0.424]},
                          index=["previous_hour", "previous_day"])
m6_written

,mae_kwh,rmse_kwh
previous_hour,0.158,0.180
previous_day,0.281,0.424


## Compare the three results

In [47]:
five_check = pd.DataFrame({
    "hand_equals_m4": (hand - m4_five).abs().max(axis=1) < 1e-12,
    "written_equals_m4": (m6_written - m4_five.round(3)).abs().max(axis=1) < 1e-9})
five_check

,hand_equals_m4,written_equals_m4
previous_hour,True,True
previous_day,True,True


## Five-row check note

**Date:** 14 November 2026. **Checked by:** M6 (Hassan Alkhamis) with M4 (Moamen Rabah).

**Rows (home, target hour UTC):** MAC000112 2013-09-06 16:00; MAC002339 2013-10-07 17:00; MAC003520 2013-10-26 15:00; MAC003814 2013-09-02 19:00; MAC005460 2013-09-05 04:00. All five are VALIDATION rows scored by both baselines.

| Rule | M6 by hand | M4 formulas | Match |
|---|---|---|---|
| previous_hour | MAE 0.158, RMSE 0.180 kWh | MAE 0.158, RMSE 0.180 kWh | True |
| previous_day | MAE 0.281, RMSE 0.424 kWh | MAE 0.281, RMSE 0.424 kWh | True |

**Match: True.** The error formulas used for `validation_scores.csv` give the same results as the hand calculation.